In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

In [ ]:
_dataset = datasets.ImageFolder(root='./datasets/cifar10/train', transform=transforms.ToTensor())
_loader  = DataLoader(_dataset)

mean   = torch.zeros(3)
sq_mean = torch.zeros(3)
n_pixels = 0

with torch.no_grad():
    for images, _ in _loader:
        b, c, h, w = images.shape
        n_pixels += b * h * w
        mean     += images.sum(dim=[0, 2, 3])
        sq_mean  += (images ** 2).sum(dim=[0, 2, 3])

mean  /= n_pixels
std    = (sq_mean / n_pixels - mean ** 2).sqrt()

MEAN = tuple(round(float(i), 4) for i in mean)
STD  = tuple(round(float(i), 4) for i in std)

MEAN, STD

In [ ]:
train_transform = transforms.Compose([
    # Data Augmentation: 학습 데이터를 인위적으로 변경하여 다양성을 높인다.

    # 상하좌우 64px zero-padding 후 랜덤 크롭, 물체 위치가 조금씩 달라진다.
    transforms.RandomCrop(415, padding=64),

    # 50% 확률로 좌우 반전
    transforms.RandomHorizontalFlip(),
    
    # 밝기 및 대비 ±20% 랜덤 변화
    transforms.ColorJitter(brightness=0.2, contrast=0.2),

    # PIL Image, Numpy array -> PyTorch Tensor
    transforms.ToTensor(),
    
    # 채널별 정규화: (pixel - mean) / std
    transforms.Normalize(MEAN, STD),
])

In [ ]:
# ImageFolder로 class 폴더 자동 인식
dataset = datasets.ImageFolder(root='./datasets/pizza_not_pizza', transform=train_transform)
train_dataset, test_dataset = random_split(dataset, [0.8, 0.2])  # train, test 분리

# DataLoader로 dataset을 mini-batch 단위로 묶어서 제공 
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=8, shuffle=False)

print(dataset.classes)
print(dataset.class_to_idx)

In [ ]:
print(len(train_dataset))
print(len(test_dataset))

In [ ]:
labels = {
  0: 'not_pizza',
  1: 'pizza'
}

pre = datasets.ImageFolder(root='./datasets/pizza_not_pizza', transform=transforms.ToTensor())

figure = plt.figure(figsize=(8, 8))
for i in range(1, 10):
  idx = torch.randint(len(pre), size=(1, )).item()
  img, label = pre[idx]
  figure.add_subplot(3, 3, i)
  plt.title(labels[label])
  plt.axis('off')
  plt.imshow(img.permute(1, 2, 0))

plt.show()